# 03 · Download and preprocess AERONET aerosol inversions — Hybrid scans, Level 2.0 (Version 3), daily averages

**What this notebook does** — the three steps of notebook 01, for the inversion product:

1. **Download** the NASA AERONET bulk archive (`.tar.gz`) into `DATA/RAW/` if it is not already there, extract it, and delete the archive.
2. **Read** every station file (`*.all`, one per AERONET site) and **merge** them into a single `pandas.DataFrame`.
3. **Export** one agent-ready Parquet table to `DATA/PROCESSED/`.

**Dataset**: AERONET **Inversions, Hybrid Products (Except Phase Functions)**, **Level 2.0**, **Version 3**, **Daily averages**, all sites —
`https://aeronet.gsfc.nasa.gov/data_push/V3/INV_HYB/INV_HYB_Level2_Daily_V3.tar.gz` (~64 MB archive → ~240 MB extracted, ~504 sites, one row per site × day, from 2014).

**What an inversion is** (Dubovik & King 2000; Sinyuk et al. 2020): the sky radiance measured along a scan at 440, 675, 870 and 1020 nm is inverted,
together with the coincident AOD, into the column aerosol's volume size distribution (22 radius bins, 0.05–15 µm), complex refractive index and
fraction of spherical particles; from these come single scattering albedo, absorption AOD, asymmetry factor, fine/coarse extinction, lidar and
depolarization ratios, and radiative fluxes/forcing. The **hybrid scan** is a sky-scan geometry of the newer Model-T Cimel instruments (since ~2014)
that reaches scattering angles of 75° + solar zenith angle, so Level 2.0 absorption retrievals are possible down to a solar zenith angle of 25°
(the classic almucantar needs ≥ 50°). *Except Phase Functions*: every product but the scattering phase function (which AERONET ships as a separate
archive, `INV_HYB_PFN_...`). One row = the daily mean of that day's Level 2.0 hybrid retrievals; `N[...]` = how many retrievals went into it.


In [1]:
from pathlib import Path
import re
import tarfile
import time

import numpy as np
import pandas as pd
import requests
from tqdm.auto import tqdm

/home/ubuntu/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
ARCHIVE = "INV_HYB_Level2_Daily_V3.tar.gz"
URL = "https://aeronet.gsfc.nasa.gov/data_push/V3/INV_HYB/" + ARCHIVE

In [3]:
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "process_data":          # this notebook lives in notebooks/process_data/
    PROJECT_ROOT = PROJECT_ROOT.parent.parent

In [4]:
RAW_DIR       = PROJECT_ROOT / "DATA" / "RAW"
PROCESSED_DIR = PROJECT_ROOT / "DATA" / "PROCESSED"
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

In [5]:
TARBALL     = RAW_DIR / ARCHIVE                                  # deleted after extraction
EXTRACT_DIR = RAW_DIR / ARCHIVE.replace(".tar.gz", "")           # station files land here
OUT_PARQUET = PROCESSED_DIR / "AERONET_INV_Hybrid_L2_Daily_V3.parquet"

In [6]:
print(f"URL          : {URL}")
print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"EXTRACT_DIR  : {EXTRACT_DIR}")
print(f"OUT_PARQUET  : {OUT_PARQUET}")

URL          : https://aeronet.gsfc.nasa.gov/data_push/V3/INV_HYB/INV_HYB_Level2_Daily_V3.tar.gz
PROJECT_ROOT : /home/ubuntu/ai_for_aeronet
EXTRACT_DIR  : /home/ubuntu/ai_for_aeronet/DATA/RAW/INV_HYB_Level2_Daily_V3
OUT_PARQUET  : /home/ubuntu/ai_for_aeronet/DATA/PROCESSED/AERONET_INV_Hybrid_L2_Daily_V3.parquet


## 1 · Download (if needed) → extract → delete the archive


In [7]:
def download(url: str, dest: Path, chunk_size: int = 1 << 20) -> None:
    tmp = dest.with_name(dest.name + ".part")
    with requests.get(url, stream=True, timeout=120) as r:
        r.raise_for_status()
        total = int(r.headers.get("Content-Length", 0)) or None
        with open(tmp, "wb") as fh, tqdm(total=total, unit="B", unit_scale=True, desc=dest.name) as bar:
            for part in r.iter_content(chunk_size):
                fh.write(part)
                bar.update(len(part))
    tmp.replace(dest)


def extract_tar_gz(archive: Path, dest: Path) -> None:
    dest.mkdir(parents=True, exist_ok=True)
    with tarfile.open(archive, "r:gz") as tf:
        try:
            tf.extractall(dest, filter="data")      
        except TypeError:                    
            tf.extractall(dest)


station_files = sorted(EXTRACT_DIR.rglob("*.all")) if EXTRACT_DIR.exists() else []

if station_files:
    print(f"Already extracted: {len(station_files)} station files in {EXTRACT_DIR}  →  skipping download.")
else:
    if TARBALL.exists():
        print(f"Found {TARBALL.name} ({TARBALL.stat().st_size / 1e6:,.0f} MB)  →  skipping download.")
    else:
        print(f"Downloading {URL}")
        t0 = time.time()
        download(URL, TARBALL)
        print(f"  done in {time.time() - t0:,.0f} s  ({TARBALL.stat().st_size / 1e6:,.0f} MB)")

    print(f"Extracting → {EXTRACT_DIR}")
    extract_tar_gz(TARBALL, EXTRACT_DIR)
    TARBALL.unlink()
    print(f"Deleted {TARBALL.name}")
    station_files = sorted(EXTRACT_DIR.rglob("*.all"))

print(f"{len(station_files):,} station files ready")

Already extracted: 504 station files in /home/ubuntu/ai_for_aeronet/DATA/RAW/INV_HYB_Level2_Daily_V3  →  skipping download.
504 station files ready


## 2 · Read every station file and merge

Each `*.all` file is one AERONET site. The header is laid out like the AOD files, except that the contact line starts with `Daily Averages,`
and the first CSV column is the site name:

```
line 1  AERONET Version 3
line 2  <site name>
line 3  Version 3: Hybrid Level 2.0 Inversion
line 4  (description)
line 5  Daily Averages,Contact: PI=<name>; PI Email=<email>
line 6  UNITS can be found at,,, https://aeronet.gsfc.nasa.gov/new_web/units.html
line 7  <CSV header>   241 columns: AERONET_Site, Date(dd:mm:yyyy), Time(hh:mm:ss), ..., Latitude(Degrees), Longitude(Degrees), Elevation(m), Level, Retrieval_Measurement_Scan_Type
line 8+ <CSV rows>, missing values are -999
```

The same `read_station` as notebook 01 reads it (the PI regex searches line 5).


In [8]:
MISSING = -999.0

def read_station(path: Path) -> pd.DataFrame:
    # Parse one AERONET .all file -> DataFrame (raw AERONET column names, -999 -> NaN,
    # plus `site`, `PI`, `PI_Email` taken from the file header).
    with open(path, encoding="latin-1") as fh:
        header = [next(fh).rstrip("\n") for _ in range(6)]
    site = header[1].strip()
    m = re.search(r"PI=(.*?);\s*PI Email=(.*)", header[4])
    pi, pi_email = (m.group(1).strip(), m.group(2).strip()) if m else (None, None)

    df = pd.read_csv(path, skiprows=6, encoding="latin-1", low_memory=False)
    df = df.mask(df == MISSING)        # -999 → NaN in one step (strings never equal -999; count columns never contain it, so they stay integer)
    df.insert(0, "site", site)
    df["PI"] = pi
    df["PI_Email"] = pi_email
    return df

In [9]:
frames, failures = [], []
t0 = time.time()
for p in tqdm(station_files, desc="Reading stations"):
    try:
        frames.append(read_station(p))
    except Exception as exc:            
        failures.append((p.name, repr(exc)))

aeronet_raw = pd.concat(frames, ignore_index=True)
del frames

print(f"Merged {len(station_files) - len(failures):,} stations → {aeronet_raw.shape[0]:,} rows × {aeronet_raw.shape[1]} columns "
      f"in {time.time() - t0:,.0f} s")
if failures:
    print(f"\n{len(failures)} file(s) failed to parse:")
    for name, err in failures:
        print(f"  {name}: {err}")

Reading stations:   0%|          | 0/504 [00:00<?, ?it/s]

Reading stations:   1%|          | 4/504 [00:00<00:13, 38.30it/s]

Reading stations:   3%|▎         | 14/504 [00:00<00:06, 70.34it/s]

Reading stations:   4%|▍         | 21/504 [00:00<00:07, 64.66it/s]

Reading stations:   6%|▌         | 28/504 [00:00<00:09, 48.00it/s]

...

Reading stations:  98%|█████████▊| 495/504 [00:13<00:00, 27.58it/s]

Reading stations:  99%|█████████▉| 499/504 [00:13<00:00, 27.30it/s]

Reading stations: 100%|█████████▉| 503/504 [00:13<00:00, 29.45it/s]

Reading stations: 100%|██████████| 504/504 [00:13<00:00, 36.84it/s]

Merged 504 stations → 163,243 rows × 340 columns in 17 s


In [10]:
aeronet_raw

,site,AERONET_Site,Date(dd:mm:yyyy),Time(hh:mm:ss),Day_of_Year,Day_of_Year(fraction),AOD_Coincident_Input[443nm],AOD_Coincident_Input[667nm],AOD_Coincident_Input[865nm],AOD_Coincident_Input[1020nm],...,N[Spectral_Flux_Up[870nm]],N[Spectral_Flux_Diffuse[440nm]],N[Spectral_Flux_Diffuse[675nm]],N[Spectral_Flux_Diffuse[870nm]],N[Lidar_Ratio[440nm]],N[Lidar_Ratio[675nm]],N[Lidar_Ratio[870nm]],N[Depolarization_Ratio[440nm]],N[Depolarization_Ratio[675nm]],N[Depolarization_Ratio[870nm]]
0,AAOT,AAOT,10:02:2018,12:00:00,41,41.0,0.125570,0.075919,0.048586,0.036544,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,AAOT,AAOT,11:02:2018,12:00:00,42,42.0,0.248580,0.155883,0.101749,0.075143,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,AAOT,AAOT,14:02:2018,12:00:00,45,45.0,0.136761,0.073338,0.045470,0.032980,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,AAOT,AAOT,24:02:2018,12:00:00,55,55.0,0.177229,0.116705,0.085278,0.070928,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,AAOT,AAOT,08:03:2018,12:00:00,67,67.0,0.192351,0.123672,0.081742,0.060059,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
163238,Zugspitze_KIT,Zugspitze_KIT,13:06:2026,12:00:00,164,164.0,NaN,NaN,NaN,0.010456,...,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
163239,Zugspitze_KIT,Zugspitze_KIT,19:06:2026,12:00:00,170,170.0,NaN,NaN,NaN,0.012987,...,6.0,6.0,6.0,6.0,0.0,0.0,0.0,0.0,0.0,0.0
163240,Zugspitze_KIT,Zugspitze_KIT,20:06:2026,12:00:00,171,171.0,NaN,NaN,NaN,0.024115,...,5.0,5.0,5.0,5.0,0.0,0.0,0.0,0.0,0.0,0.0
163241,Zugspitze_KIT,Zugspitze_KIT,22:06:2026,12:00:00,173,173.0,NaN,NaN,NaN,0.025571,...,3.0,3.0,3.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0


In [11]:
aeronet_raw.columns.tolist()

['site',
 'AERONET_Site',
 'Date(dd:mm:yyyy)',
 'Time(hh:mm:ss)',
 'Day_of_Year',
 'Day_of_Year(fraction)',
 'AOD_Coincident_Input[443nm]',
 'AOD_Coincident_Input[667nm]',
 'AOD_Coincident_Input[865nm]',
 'AOD_Coincident_Input[1020nm]',
 'Angstrom_Exponent_440-870nm_from_Coincident_Input_AOD',
 'AOD_Extinction-Total[443nm]',
 'AOD_Extinction-Total[667nm]',
 'AOD_Extinction-Total[865nm]',
 'AOD_Extinction-Total[1020nm]',
 'AOD_Extinction-Fine[443nm]',
 'AOD_Extinction-Fine[667nm]',
 'AOD_Extinction-Fine[865nm]',
 'AOD_Extinction-Fine[1020nm]',
 'AOD_Extinction-Coarse[443nm]',
 'AOD_Extinction-Coarse[667nm]',
 'AOD_Extinction-Coarse[865nm]',
 'AOD_Extinction-Coarse[1020nm]',
 'Extinction_Angstrom_Exponent_440-870nm-Total',
 'Single_Scattering_Albedo[443nm]',
 'Single_Scattering_Albedo[667nm]',
 'Single_Scattering_Albedo[865nm]',
 'Single_Scattering_Albedo[1020nm]',
 'Absorption_AOD[443nm]',
 'Absorption_AOD[667nm]',
 'Absorption_AOD[865nm]',
 'Absorption_AOD[1020nm]',
 'Absorption_Angstr

## 3 · Tidy into an agent-ready table

* **site, location and level columns get the names of the AOD tables** (`AERONET_Site` → `AERONET_Site_Name`, `Latitude(Degrees)` → `Site_Latitude_Degrees`, …, `Level` → `Data_Quality_Level`), so the four tables join on site name and date
* **the 22 size-distribution bins** are headed by their radius in µm (`0.050000` … `15.000000`): they become `dVdlnr_0_050000um` … `dVdlnr_15_000000um` (the quantity is dV/dln r in µm³/µm²; the radius keeps all its digits, with `_` for the decimal point)
* everything else keeps AERONET's own name, with `( ) [ ] -` → `_` and `(%)` → `_percent`, as in notebook 01 (`Single_Scattering_Albedo[440nm]` → `Single_Scattering_Albedo_440nm`, `AOD_Extinction-Total[440nm]` → `AOD_Extinction_Total_440nm`, `Sphericity_Factor(%)` → `Sphericity_Factor_percent`, `N[VolC-T]` → `N_VolC_T`)
* `datetime_utc`, `date`, `year`, `month` from AERONET's `dd:mm:yyyy` + `hh:mm:ss` columns, as in notebook 01
* **two filter sets**: most instruments retrieve at 440 / 675 / 870 / 1020 nm, the ocean-colour (SeaPRISM-type) instruments at 443 / 667 / 865 / 1020 nm, and each station file is headed with its own set. Merging by column name keeps both sets, so every spectral product exists twice (`Single_Scattering_Albedo_440nm` *and* `Single_Scattering_Albedo_443nm`); a site fills one of the two
* drop products that are missing at every site (all-NaN) and their count columns
* column order: site → time → location, then the products family by family in AERONET's order and, inside a family, by wavelength


In [12]:
def clean_col(name: str) -> str:
    name = re.sub(r"(\d{3})-(\d{3})_Angstrom_Exponent", r"Angstrom_Exponent_\1_\2nm", name)
    name = name.replace("[Polar]", "_Polar").replace("(%)", "_percent")
    name = re.sub(r"[\(\[]", "_", name)
    name = re.sub(r"[\)\]]", "", name)
    name = re.sub(r"[^0-9A-Za-z_]", "_", name)
    return re.sub(r"_+", "_", name).strip("_")

In [13]:
df = aeronet_raw.copy()

In [14]:
# ── the names the AOD tables use for the same things -------------------------------
df = df.rename(columns={"AERONET_Site": "AERONET_Site_Name", "Latitude(Degrees)": "Site_Latitude(Degrees)",
                        "Longitude(Degrees)": "Site_Longitude(Degrees)", "Elevation(m)": "Site_Elevation(m)",
                        "Level": "Data_Quality_Level"})

In [15]:
# ── size-distribution bins: "0.050000" -> "dVdlnr_0.050000um", and their counts "N[0.050000]" -> "N[dVdlnr_0.050000um]"
bins = {c: f"dVdlnr_{c}um" for c in df.columns if re.fullmatch(r"\d+\.\d+", c)}
print(f"{len(bins)} size bins, radius {min(bins, key=float)} … {max(bins, key=float)} µm")
bins.update({f"N[{old}]": f"N[{new}]" for old, new in bins.items()})
df = df.rename(columns=bins)

22 size bins, radius 0.050000 … 15.000000 µm


In [16]:
# AERONET stamps every daily average at 12:00:00 UTC
df["datetime_utc"] = pd.to_datetime(df["Date(dd:mm:yyyy)"] + " " + df["Time(hh:mm:ss)"], format="%d:%m:%Y %H:%M:%S")
df = df.drop(columns=["Date(dd:mm:yyyy)", "Time(hh:mm:ss)"])

df["date"]  = df["datetime_utc"].dt.normalize()
df["year"]  = df["datetime_utc"].dt.year.astype("int16")
df["month"] = df["datetime_utc"].dt.month.astype("int8")

In [17]:
df.columns = [clean_col(c) for c in df.columns]

In [18]:
# AERONET repeats the site name in every row – make sure it matches the file header, then keep the original column
mismatch = (df["AERONET_Site_Name"] != df["site"]).sum()
print(f"site-name mismatches between header and rows: {mismatch}")
df = df.drop(columns=["site"])

site-name mismatches between header and rows: 0


In [19]:
# ── drop products that no site ever has (all-NaN) -------------------------------
all_nan = [c for c in df.columns if df[c].isna().all()]
if all_nan:
    print(f"dropping {len(all_nan)} all-NaN columns: {all_nan}")
    df = df.drop(columns=all_nan)

dropping 2 all-NaN columns: ['Sphericity_Factor_percent', 'Inflection_Radius_of_Size_Distribution_um']


In [20]:
# … and their (now meaningless) observation-count columns
df = df.drop(columns=[c for c in df.columns if c.startswith("N_") and c[2:] not in df.columns])

In [21]:
# ── dtypes ----------------------------------------------------------------------
for c in df.columns:
    if c.startswith("N_"):
        df[c] = df[c].astype("Int32")
df["AERONET_Site_Name"] = df["AERONET_Site_Name"].astype("string")
for c in ("PI", "PI_Email", "Data_Quality_Level"):
    if c in df.columns:
        df[c] = df[c].astype("string")

In [22]:
# ── column order: site → time → location, then the products family by family in AERONET's order, each family
# by wavelength (440, 443, 667, 675, 865, 870, 1020 nm: the two filter sets end up side by side)
front = ["AERONET_Site_Name", "datetime_utc", "date", "year", "month", "Day_of_Year", "Day_of_Year_fraction",
         "Site_Latitude_Degrees", "Site_Longitude_Degrees", "Site_Elevation_m"]
rest  = [c for c in df.columns if c not in front]

def _family(c):     return re.sub(r"_\d+nm", "", c)                                   # Single_Scattering_Albedo_440nm -> Single_Scattering_Albedo
def _wavelength(c): m = re.search(r"_(\d+)nm", c); return int(m.group(1)) if m else 0

families = list(dict.fromkeys(_family(c) for c in rest))                               # in order of first appearance
rest = sorted(rest, key=lambda c: (families.index(_family(c)), _wavelength(c)))
df = df[front + rest]
df = df.sort_values(["AERONET_Site_Name", "datetime_utc"], ignore_index=True)

In [23]:
aeronet = df
del df
print(f"{aeronet.shape[0]:,} rows × {aeronet.shape[1]} columns")
print(list(aeronet.columns))

163,243 rows × 337 columns
['AERONET_Site_Name', 'datetime_utc', 'date', 'year', 'month', 'Day_of_Year', 'Day_of_Year_fraction', 'Site_Latitude_Degrees', 'Site_Longitude_Degrees', 'Site_Elevation_m', 'AOD_Coincident_Input_440nm', 'AOD_Coincident_Input_443nm', 'AOD_Coincident_Input_667nm', 'AOD_Coincident_Input_675nm', 'AOD_Coincident_Input_865nm', 'AOD_Coincident_Input_870nm', 'AOD_Coincident_Input_1020nm', 'Angstrom_Exponent_440_870nm_from_Coincident_Input_AOD', 'AOD_Extinction_Total_440nm', 'AOD_Extinction_Total_443nm', 'AOD_Extinction_Total_667nm', 'AOD_Extinction_Total_675nm', 'AOD_Extinction_Total_865nm', 'AOD_Extinction_Total_870nm', 'AOD_Extinction_Total_1020nm', 'AOD_Extinction_Fine_440nm', 'AOD_Extinction_Fine_443nm', 'AOD_Extinction_Fine_667nm', 'AOD_Extinction_Fine_675nm', 'AOD_Extinction_Fine_865nm', 'AOD_Extinction_Fine_870nm', 'AOD_Extinction_Fine_1020nm', 'AOD_Extinction_Coarse_440nm', 'AOD_Extinction_Coarse_443nm', 'AOD_Extinction_Coarse_667nm', 'AOD_Extinction_Coarse_6

In [24]:
aeronet

,AERONET_Site_Name,datetime_utc,date,year,month,Day_of_Year,Day_of_Year_fraction,Site_Latitude_Degrees,Site_Longitude_Degrees,Site_Elevation_m,...,N_Depolarization_Ratio_667nm,N_Depolarization_Ratio_675nm,N_Depolarization_Ratio_865nm,N_Depolarization_Ratio_870nm,N_Depolarization_Ratio_1020nm,AERONET_Instrument_Number,Data_Quality_Level,Retrieval_Measurement_Scan_Type,PI,PI_Email
0,AAOT,2018-02-10 12:00:00,2018-02-10,2018,2,41,41.0,45.3139,12.5083,10.00,...,0,<NA>,0,<NA>,0,1001,lev20,Hybrid,Giuseppe_Zibordi,giuseppe.zibordi@eoscience.eu
1,AAOT,2018-02-11 12:00:00,2018-02-11,2018,2,42,42.0,45.3139,12.5083,10.00,...,0,<NA>,0,<NA>,0,1001,lev20,Hybrid,Giuseppe_Zibordi,giuseppe.zibordi@eoscience.eu
2,AAOT,2018-02-14 12:00:00,2018-02-14,2018,2,45,45.0,45.3139,12.5083,10.00,...,0,<NA>,0,<NA>,0,1001,lev20,Hybrid,Giuseppe_Zibordi,giuseppe.zibordi@eoscience.eu
3,AAOT,2018-02-24 12:00:00,2018-02-24,2018,2,55,55.0,45.3139,12.5083,10.00,...,0,<NA>,0,<NA>,0,1001,lev20,Hybrid,Giuseppe_Zibordi,giuseppe.zibordi@eoscience.eu
4,AAOT,2018-03-08 12:00:00,2018-03-08,2018,3,67,67.0,45.3139,12.5083,10.00,...,0,<NA>,0,<NA>,0,1001,lev20,Hybrid,Giuseppe_Zibordi,giuseppe.zibordi@eoscience.eu
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
163238,Zugspitze_KIT,2026-06-13 12:00:00,2026-06-13,2026,6,164,164.0,47.4210,10.9860,2961.46,...,<NA>,0,<NA>,0,0,1246,lev20,Hybrid,Hannes_Vogelmann_and_Michal_Posyniak,hannes.vogelmann@kit.edu_and_michal.posyniak@k...
163239,Zugspitze_KIT,2026-06-19 12:00:00,2026-06-19,2026,6,170,170.0,47.4210,10.9860,2961.46,...,<NA>,0,<NA>,0,0,1246,lev20,Hybrid,Hannes_Vogelmann_and_Michal_Posyniak,hannes.vogelmann@kit.edu_and_michal.posyniak@k...
163240,Zugspitze_KIT,2026-06-20 12:00:00,2026-06-20,2026,6,171,171.0,47.4210,10.9860,2961.46,...,<NA>,0,<NA>,0,0,1246,lev20,Hybrid,Hannes_Vogelmann_and_Michal_Posyniak,hannes.vogelmann@kit.edu_and_michal.posyniak@k...
163241,Zugspitze_KIT,2026-06-22 12:00:00,2026-06-22,2026,6,173,173.0,47.4210,10.9860,2961.46,...,<NA>,0,<NA>,0,0,1246,lev20,Hybrid,Hannes_Vogelmann_and_Michal_Posyniak,hannes.vogelmann@kit.edu_and_michal.posyniak@k...


## 4 · Sanity checks


In [25]:
print(f"sites        : {aeronet['AERONET_Site_Name'].nunique():,}")
print(f"rows         : {len(aeronet):,}")
print(f"time range   : {aeronet['datetime_utc'].min()}  →  {aeronet['datetime_utc'].max()}")
print(f"duplicates   : {aeronet.duplicated(['AERONET_Site_Name', 'datetime_utc']).sum():,}  (site × datetime)")
print(f"memory       : {aeronet.memory_usage(deep=True).sum() / 1e6:,.0f} MB")

print("\nnon-null fraction per column:")
display((aeronet.notna().mean() * 100).round(1).rename("% non-null").to_frame().T)

display(aeronet.head())

sites        : 504
rows         : 163,243
time range   : 2014-06-13 12:00:00  →  2026-06-26 12:00:00
duplicates   : 0  (site × datetime)


memory       : 406 MB

non-null fraction per column:


,AERONET_Site_Name,datetime_utc,date,year,month,Day_of_Year,Day_of_Year_fraction,Site_Latitude_Degrees,Site_Longitude_Degrees,Site_Elevation_m,...,N_Depolarization_Ratio_667nm,N_Depolarization_Ratio_675nm,N_Depolarization_Ratio_865nm,N_Depolarization_Ratio_870nm,N_Depolarization_Ratio_1020nm,AERONET_Instrument_Number,Data_Quality_Level,Retrieval_Measurement_Scan_Type,PI,PI_Email
% non-null,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,...,4.7,95.3,4.7,95.3,100.0,100.0,100.0,100.0,100.0,100.0


,AERONET_Site_Name,datetime_utc,date,year,month,Day_of_Year,Day_of_Year_fraction,Site_Latitude_Degrees,Site_Longitude_Degrees,Site_Elevation_m,...,N_Depolarization_Ratio_667nm,N_Depolarization_Ratio_675nm,N_Depolarization_Ratio_865nm,N_Depolarization_Ratio_870nm,N_Depolarization_Ratio_1020nm,AERONET_Instrument_Number,Data_Quality_Level,Retrieval_Measurement_Scan_Type,PI,PI_Email
0,AAOT,2018-02-10 12:00:00,2018-02-10,2018,2,41,41.0,45.3139,12.5083,10.0,...,0,<NA>,0,<NA>,0,1001,lev20,Hybrid,Giuseppe_Zibordi,giuseppe.zibordi@eoscience.eu
1,AAOT,2018-02-11 12:00:00,2018-02-11,2018,2,42,42.0,45.3139,12.5083,10.0,...,0,<NA>,0,<NA>,0,1001,lev20,Hybrid,Giuseppe_Zibordi,giuseppe.zibordi@eoscience.eu
2,AAOT,2018-02-14 12:00:00,2018-02-14,2018,2,45,45.0,45.3139,12.5083,10.0,...,0,<NA>,0,<NA>,0,1001,lev20,Hybrid,Giuseppe_Zibordi,giuseppe.zibordi@eoscience.eu
3,AAOT,2018-02-24 12:00:00,2018-02-24,2018,2,55,55.0,45.3139,12.5083,10.0,...,0,<NA>,0,<NA>,0,1001,lev20,Hybrid,Giuseppe_Zibordi,giuseppe.zibordi@eoscience.eu
4,AAOT,2018-03-08 12:00:00,2018-03-08,2018,3,67,67.0,45.3139,12.5083,10.0,...,0,<NA>,0,<NA>,0,1001,lev20,Hybrid,Giuseppe_Zibordi,giuseppe.zibordi@eoscience.eu


## 5 · Export to `DATA/PROCESSED/`

One file: `AERONET_INV_Hybrid_L2_Daily_V3.parquet` — the full merged table, one row per site × day.


In [26]:
aeronet.to_parquet(OUT_PARQUET, index=False, engine="pyarrow", compression="zstd")
print(f"wrote {OUT_PARQUET}  ({OUT_PARQUET.stat().st_size / 1e6:,.0f} MB, {len(aeronet):,} rows × {aeronet.shape[1]} columns)")

wrote /home/ubuntu/ai_for_aeronet/DATA/PROCESSED/AERONET_INV_Hybrid_L2_Daily_V3.parquet  (90 MB, 163,243 rows × 337 columns)


### Output schema (data dictionary for the agent's skill / system prompt)

Column names are AERONET's own, with `( ) [ ] -` → `_` and `(%)` → `_percent`. Wavelengths `<λ>`: 440, 675, 870, 1020 nm (the sky-radiance channels of most instruments) **or** 443, 667, 865, 1020 nm (ocean-colour instruments): every spectral product has both sets of columns and a site fills one of them.
Units from <https://aeronet.gsfc.nasa.gov/new_web/units.html> and the AERONET inversion-products document.

| column | original AERONET header | unit | meaning |
|---|---|---|---|
| `AERONET_Site_Name` | `AERONET_Site` | – | site name (string) |
| `datetime_utc`, `date`, `year`, `month` | *derived from* `Date(dd:mm:yyyy)` + `Time(hh:mm:ss)` | UTC | daily averages are stamped 12:00:00 UTC |
| `Day_of_Year`, `Day_of_Year_fraction` | `Day_of_Year`, `Day_of_Year(fraction)` | day | 1–366 |
| `Site_Latitude_Degrees`, `Site_Longitude_Degrees`, `Site_Elevation_m` | `Latitude(Degrees)`, `Longitude(Degrees)`, `Elevation(m)` | degrees, m | site location |
| `AOD_Coincident_Input_<λ>nm` | `AOD_Coincident_Input[<λ>nm]` | unitless | the measured (direct-sun) AOD that went into the retrieval |
| `Angstrom_Exponent_440_870nm_from_Coincident_Input_AOD` | `Angstrom_Exponent_440-870nm_from_Coincident_Input_AOD` | unitless | Ångström exponent of that input AOD |
| `AOD_Extinction_Total_<λ>nm`, `_Fine_`, `_Coarse_` | `AOD_Extinction-Total[<λ>nm]` … | unitless | extinction AOD of the retrieved aerosol, total and split at the fine/coarse boundary |
| `Extinction_Angstrom_Exponent_440_870nm_Total` | `Extinction_Angstrom_Exponent_440-870nm-Total` | unitless | Ångström exponent of the retrieved extinction |
| `Single_Scattering_Albedo_<λ>nm` | `Single_Scattering_Albedo[<λ>nm]` | unitless (0–1) | scattering / extinction; 1 = no absorption |
| `Absorption_AOD_<λ>nm` | `Absorption_AOD[<λ>nm]` | unitless | absorption optical depth = AOD × (1 − SSA) |
| `Absorption_Angstrom_Exponent_440_870nm` | `Absorption_Angstrom_Exponent_440-870nm` | unitless | spectral slope of the absorption AOD |
| `Refractive_Index_Real_Part_<λ>nm`, `Refractive_Index_Imaginary_Part_<λ>nm` | `Refractive_Index-Real_Part[<λ>nm]`, `…Imaginary_Part[…]` | unitless | complex refractive index (real 1.33–1.6, imaginary 0.0005–0.5) |
| `Asymmetry_Factor_Total_<λ>nm`, `_Fine_`, `_Coarse_` | `Asymmetry_Factor-Total[<λ>nm]` … | unitless (−1–1) | mean cosine of the scattering angle |
| *(dropped)* | `Sphericity_Factor(%)` | % | always -999 in the hybrid daily product: dropped as all-NaN |
| `dVdlnr_<r>um` (22 columns) | `0.050000` … `15.000000` | µm³/µm² | volume size distribution dV/dln r at bin radius r µm (`_` stands for the decimal point: `dVdlnr_0_050000um` = 0.05 µm) |
| *(dropped)* | `Inflection_Radius_of_Size_Distribution(um)` | µm | fine/coarse separation radius (minimum of dV/dln r between 0.439 and 0.992 µm): always -999 in the hybrid daily product, dropped as all-NaN |
| `VolC_T`, `VolC_F`, `VolC_C` | `VolC-T`, `VolC-F`, `VolC-C` | µm³/µm² | volume concentration, total / fine / coarse |
| `REff_T/F/C`, `VMR_T/F/C`, `Std_T/F/C` | `REff-T` …, `VMR-T` …, `Std-T` … | µm, µm, unitless | effective radius, volume median radius, standard deviation of ln r |
| `Flux_Down_BOA/TOA`, `Flux_Up_BOA/TOA`, `Diffuse_BOA/TOA` | `Flux_Down(BOA)` … | W/m² | broadband (0.2–4 µm) fluxes at the bottom / top of the atmosphere |
| `Rad_Forcing_BOA/TOA`, `Forcing_Eff_BOA/TOA` | `Rad_Forcing(BOA)` …, `Forcing_Eff(BOA)` … | W/m², W/m² per unit AOD(550) | aerosol radiative forcing (negative = cooling) and forcing efficiency |
| `Minimum/Maximum_Altitude_For_Flux_Calculations_km` | `…(km)` | km | flux-calculation layer |
| `Spectral_Flux_Down/Up/Diffuse_<λ>nm` | `Spectral_Flux_Down[<λ>nm]` … | W/m²/µm | spectral fluxes |
| `Lidar_Ratio_<λ>nm` | `Lidar_Ratio[<λ>nm]` | sr | extinction-to-backscatter ratio |
| `Depolarization_Ratio_<λ>nm` | `Depolarization_Ratio[<λ>nm]` | unitless | linear depolarization ratio (high = non-spherical) |
| `N_<column>` | `N[<column>]` | count | number of Level 2.0 retrievals averaged into that day's value (0 ⇒ NaN) |
| `Data_Quality_Level`, `Retrieval_Measurement_Scan_Type` | `Level`, `Retrieval_Measurement_Scan_Type` | – | always `lev20`, `Hybrid` |
| `AERONET_Instrument_Number`, `PI`, `PI_Email` | header | – | instrument id, site principal investigator |
